In [1]:
import sys
sys.path.append('..')

In [2]:
from src.slices_pipeline.encoder import *
from src.slices_pipeline.loader import *

In [ ]:
from pymatgen.core.composition import Composition
import re
import pandas as pd

def update_result(df):
    def count_atoms(formula):
        try:
            comp = Composition(formula)
            return comp.num_atoms
        except Exception:
            return None

    df['atom_count'] = df['pretty_formula'].apply(count_atoms)
    return df

def read_results(file):
    rows = []
    pattern = r'^(?P<status>E|x)?\s*(?P<idx>\d+):\s*(?P<material>[a-zA-Z]+-\d+)\s*---\s*(?P<time>[\d\.eE+-]+)'

    for line in open(file).read().strip().split('\n'):
        m = re.match(pattern, line)
        if m:
            rows.append({
                'status': m.group('status') or '',
                'idx': int(m.group('idx')),
                'material_id': m.group('material'),
                'time': float(m.group('time')),
            })

    df = pd.DataFrame(rows)

    split = pd.concat([read_split_df("train"), read_split_df("test"), read_split_df("val")])

    # result = pd.merge(df, split, on='material_id', how='inner')
    result = pd.merge(df, split, on='material_id', how='left')

    return update_result(result), df

In [5]:
tests, res = read_results("results/results-full.txt")
tests

,status,idx,material_id,time,Unnamed: 0,formation_energy_per_atom,band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup.number,atom_count
0,,0,mp-1221227,3.763618,37228,-1.637460,0.2133,Na3MnCoNiO6,0.043001,"['Co', 'Mn', 'Na', 'Ni', 'O']",# generated using pymatgen\ndata_Na3MnCoNiO6\n...,8,12.0
1,,1,mp-974729,1.464253,19480,-0.314759,0.0000,Nd(Al2Cu)4,0.000000,"['Al', 'Cu', 'Nd']",# generated using pymatgen\ndata_Nd(Al2Cu)4\n_...,139,13.0
2,,2,mp-1185360,0.320010,29624,-0.193761,0.0000,LiMnIr2,0.018075,"['Ir', 'Li', 'Mn']",# generated using pymatgen\ndata_LiMnIr2\n_sym...,225,4.0
3,E,3,mp-1188861,0.702790,38633,-0.584694,3.8556,LiCSN,0.048847,"['C', 'Li', 'N', 'S']",# generated using pymatgen\ndata_LiCSN\n_symme...,62,4.0
4,,4,mp-677272,3.654627,10889,-2.474759,0.4707,La2EuS4,0.000000,"['Eu', 'La', 'S']",# generated using pymatgen\ndata_La2EuS4\n_sym...,122,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,x,164,mp-1215267,5.624179,35677,-3.726962,0.0000,ZrU4O10,0.036117,"['O', 'U', 'Zr']",# generated using pymatgen\ndata_ZrU4O10\n_sym...,166,15.0
165,,165,mp-972246,5.045741,9982,-0.404259,0.0000,Tb2ReC2,0.000000,"['C', 'Re', 'Tb']",# generated using pymatgen\ndata_Tb2ReC2\n_sym...,62,5.0
166,,166,mp-1113274,1.016763,41090,-3.121153,0.0000,Rb2CeAgF6,0.058826,"['Ag', 'Ce', 'F', 'Rb']",# generated using pymatgen\ndata_Rb2CeAgF6\n_s...,225,10.0
167,,167,mp-755786,3.214105,37916,-1.721172,0.2082,Li5Fe3(NiO5)2,0.044943,"['Fe', 'Li', 'Ni', 'O']",# generated using pymatgen\ndata_Li5Fe3(NiO5)2...,2,20.0


In [5]:
res[res.idx.duplicated()].groupby("idx").count()

,status,material_id,time
idx,,,
0,2,2,2
1,2,2,2
2,2,2,2
3,2,2,2
4,2,2,2
...,...,...,...
18297,1,1,1
18298,3,3,3
18299,4,4,4


In [6]:
res[res.idx == 0]

,status,idx,material_id,time
3,,0,mp-1221227,15.862125
32333,E,0,mp-10009,0.621733
41735,,0,mp-865981,1.698157


In [7]:
tests.material_id.unique(), tests.material_id.unique().shape[0]

(array(['mp-573037', 'mp-1184907', 'mp-1227282', ..., 'mp-1207908',
        'mp-637600', 'mp-643385'], dtype=object),
 45208)

In [8]:
non_unique = tests[tests.duplicated('material_id', keep=False)]
non_unique.sort_values(by="material_id")

,status,idx,material_id,time,Unnamed: 0,formation_energy_per_atom,band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup.number,atom_count
28339,,15361,mp-10000,1.024807,15000,-1.195884,0.0000,Hf2S,0.000000,"['Hf', 'S']",# generated using pymatgen\ndata_Hf2S\n_symmet...,194,3.0
28350,,15361,mp-10000,1.819313,15000,-1.195884,0.0000,Hf2S,0.000000,"['Hf', 'S']",# generated using pymatgen\ndata_Hf2S\n_symmet...,194,3.0
28351,,15361,mp-10000,2.160675,15000,-1.195884,0.0000,Hf2S,0.000000,"['Hf', 'S']",# generated using pymatgen\ndata_Hf2S\n_symmet...,194,3.0
28246,E,2016,mp-1077107,0.415750,21533,-0.636121,1.2835,HgO,0.001932,"['Hg', 'O']",# generated using pymatgen\ndata_HgO\n_symmetr...,154,2.0
28273,E,2016,mp-1077107,1.458125,21533,-0.636121,1.2835,HgO,0.001932,"['Hg', 'O']",# generated using pymatgen\ndata_HgO\n_symmetr...,154,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
28258,,10919,mp-867877,1.004567,539,-0.371367,0.0000,LiCdAu2,0.000000,"['Li', 'Cd', 'Au']",# generated using pymatgen\ndata_LiCdAu2\n_sym...,225,4.0
28274,,10919,mp-867877,2.233990,539,-0.371367,0.0000,LiCdAu2,0.000000,"['Li', 'Cd', 'Au']",# generated using pymatgen\ndata_LiCdAu2\n_sym...,225,4.0
28279,,5135,mp-975079,4.902698,27968,-0.017600,0.0000,MnZn3,0.013461,"['Mn', 'Zn']",# generated using pymatgen\ndata_MnZn3\n_symme...,194,4.0
28314,,5135,mp-975079,6.549609,27968,-0.017600,0.0000,MnZn3,0.013461,"['Mn', 'Zn']",# generated using pymatgen\ndata_MnZn3\n_symme...,194,4.0


In [9]:
non_unique.groupby("material_id").sum().shape[0]

28

In [10]:
tests

,status,idx,material_id,time,Unnamed: 0,formation_energy_per_atom,band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup.number,atom_count
0,E,3950,mp-573037,4.902676,9885,-2.850817,1.9445,Tb2MnNiO6,0.000000,"['Mn', 'Ni', 'O', 'Tb']",# generated using pymatgen\ndata_Tb2MnNiO6\n_s...,14,10.0
1,,2826,mp-1184907,9.641253,42128,0.035988,0.0000,K3Cd,0.065348,"['Cd', 'K']",# generated using pymatgen\ndata_K3Cd\n_symmet...,139,4.0
2,,8478,mp-1227282,14.334577,43731,-0.115920,0.0000,Be4SiNi,0.073204,"['Be', 'Ni', 'Si']",# generated using pymatgen\ndata_Be4SiNi\n_sym...,216,6.0
3,,0,mp-1221227,15.862125,37228,-1.637460,0.2133,Na3MnCoNiO6,0.043001,"['Co', 'Mn', 'Na', 'Ni', 'O']",# generated using pymatgen\ndata_Na3MnCoNiO6\n...,8,12.0
4,,14130,mp-1018818,15.927019,25506,-2.165928,0.4133,NdTeCl,0.009418,"['Nd', 'Te', 'Cl']",# generated using pymatgen\ndata_NdTeCl\n_symm...,129,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
45274,,10155,mp-979028,0.490691,29486,-0.049365,0.0000,Sm2Mg,0.017045,"['Mg', 'Sm']",# generated using pymatgen\ndata_Sm2Mg\n_symme...,139,3.0
45275,,10124,mp-772624,4.923565,44952,-1.423574,2.7400,Al2CO,0.076504,"['Al', 'C', 'O']",# generated using pymatgen\ndata_Al2CO\n_symme...,33,4.0
45276,,10034,mp-1207908,12.965023,35156,-2.090231,0.3697,V4CuO11,0.034206,"['Cu', 'O', 'V']",# generated using pymatgen\ndata_V4CuO11\n_sym...,8,16.0
45277,x,9641,mp-637600,14.853575,10881,-1.916371,0.0013,Gd6CoBr10,0.000000,"['Br', 'Co', 'Gd']",# generated using pymatgen\ndata_Gd6CoBr10\n_s...,2,17.0


In [11]:
tests[tests.pretty_formula.isna()]

,status,idx,material_id,time,Unnamed: 0,formation_energy_per_atom,band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup.number,atom_count


In [12]:
non_unique_values = tests['material_id'][tests['material_id'].duplicated(keep=False)].unique()
non_unique_values,non_unique_values.shape[0]

(array(['mp-1079698', 'mp-1227051', 'mp-1077107', 'mp-27641', 'mp-1207025',
        'mp-13376', 'mp-1221914', 'mp-867877', 'mp-7572', 'mp-2426',
        'mp-1222601', 'mp-754883', 'mp-753275', 'mp-6192', 'mp-975079',
        'mp-22862', 'mp-1188878', 'mp-761710', 'mp-1079550', 'mp-11481',
        'mp-867785', 'mp-10000', 'mp-1080531', 'mp-23319', 'mp-1186961',
        'mp-11487', 'mp-1222288', 'mp-156'], dtype=object),
 28)

In [13]:
all = pd.concat([read_split_df("train"), read_split_df("test"), read_split_df("val")])
all

,Unnamed: 0,material_id,formation_energy_per_atom,band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup.number
0,37228,mp-1221227,-1.637460,0.2133,Na3MnCoNiO6,0.043001,"['Co', 'Mn', 'Na', 'Ni', 'O']",# generated using pymatgen\ndata_Na3MnCoNiO6\n...,8
1,19480,mp-974729,-0.314759,0.0000,Nd(Al2Cu)4,0.000000,"['Al', 'Cu', 'Nd']",# generated using pymatgen\ndata_Nd(Al2Cu)4\n_...,139
2,29624,mp-1185360,-0.193761,0.0000,LiMnIr2,0.018075,"['Ir', 'Li', 'Mn']",# generated using pymatgen\ndata_LiMnIr2\n_sym...,225
3,38633,mp-1188861,-0.584694,3.8556,LiCSN,0.048847,"['C', 'Li', 'N', 'S']",# generated using pymatgen\ndata_LiCSN\n_symme...,62
4,10889,mp-677272,-2.474759,0.4707,La2EuS4,0.000000,"['Eu', 'La', 'S']",# generated using pymatgen\ndata_La2EuS4\n_sym...,122
...,...,...,...,...,...,...,...,...,...
9042,21009,mp-1023925,-1.152798,1.6970,WS2,0.001173,"['S', 'W']",# generated using pymatgen\ndata_WS2\n_symmetr...,164
9043,31626,mp-1187764,-0.788772,0.0000,Y2ZnPt,0.022813,"['Pt', 'Y', 'Zn']",# generated using pymatgen\ndata_Y2ZnPt\n_symm...,225
9044,20673,mp-1219588,-2.910913,1.9239,RbMgCoF6,0.000710,"['Co', 'F', 'Mg', 'Rb']",# generated using pymatgen\ndata_RbMgCoF6\n_sy...,74
9045,5349,mp-3589,-2.764644,7.2758,BPO4,0.000000,"['B', 'P', 'O']",# generated using pymatgen\ndata_BPO4\n_symmet...,82


In [ ]:
lost = all[~all.material_id.isin(tests.material_id)]
lost.shape[0]

21

In [ ]:
lost.material_id

9420       mp-10018
9421     mp-1211359
9422     mp-1209679
9423     mp-1020059
9424     mp-1189429
9425      mp-756309
9426     mp-1220565
9427       mp-38816
9428       mp-10867
9429        mp-4195
9442     mp-1273854
9443     mp-1224805
9564      mp-676762
9573     mp-1102240
9574     mp-1113559
10035      mp-19376
10036      mp-10091
10045       mp-6509
10046     mp-975522
10047    mp-1225893
10089     mp-862832
Name: material_id, dtype: object

In [ ]:
lost.material_id.to_csv('lost.txt', index=False, header=False)


In [17]:
all.material_id.unique().shape[0]

45229

## Добавляем потерянные строчки

In [ ]:
df_sorted = res.sort_values(by='time')
res = df_sorted.drop_duplicates(subset='material_id', keep='first')
res.drop(columns=['idx'], inplace=True)

/tmp/ipykernel_11326/907496822.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res.drop(columns=['idx'], inplace=True)


In [21]:
_, lost = read_results("results-lost-true.txt")
lost

,status,idx,material_id,time
0,E,9420,mp-10018,0.123928
1,E,9421,mp-1211359,5.697293
2,,9422,mp-1209679,0.789871
3,,9423,mp-1020059,0.620733
4,,9424,mp-1189429,1.205847
5,,9425,mp-756309,2.557689
6,,9426,mp-1220565,1.129870
7,E,9427,mp-38816,0.633848
8,,9428,mp-10867,0.267304
9,E,9429,mp-4195,0.241615


In [19]:
res

,status,material_id,time
26747,E,mp-611836,0.011747
5457,E,mp-22885,0.013471
42028,E,mp-571436,0.013493
18985,E,mp-29803,0.013640
30819,E,mp-1008394,0.013818
...,...,...,...
2239,E,mp-559415,87.655424
2230,E,mp-1223305,89.773011
1777,E,mp-6036,90.108153
22381,E,mp-1183057,94.462643


In [23]:
final = pd.concat([lost, res])
final

,status,idx,material_id,time
0,E,9420.0,mp-10018,0.123928
1,E,9421.0,mp-1211359,5.697293
2,,9422.0,mp-1209679,0.789871
3,,9423.0,mp-1020059,0.620733
4,,9424.0,mp-1189429,1.205847
...,...,...,...,...
2239,E,NaN,mp-559415,87.655424
2230,E,NaN,mp-1223305,89.773011
1777,E,NaN,mp-6036,90.108153
22381,E,NaN,mp-1183057,94.462643


In [28]:
final.drop(columns=["idx"], inplace=True)
final

,status,material_id,time
0,E,mp-10018,0.123928
1,E,mp-1211359,5.697293
2,,mp-1209679,0.789871
3,,mp-1020059,0.620733
4,,mp-1189429,1.205847
...,...,...,...
2239,E,mp-559415,87.655424
2230,E,mp-1223305,89.773011
1777,E,mp-6036,90.108153
22381,E,mp-1183057,94.462643


In [29]:
final.to_csv("../mp20-results.csv")

In [31]:
s = pd.read_csv("../mp20-results.csv").fillna(' ')
s

,Unnamed: 0,status,material_id,time
0,0,E,mp-10018,0.123928
1,1,E,mp-1211359,5.697293
2,2,,mp-1209679,0.789871
3,3,,mp-1020059,0.620733
4,4,,mp-1189429,1.205847
...,...,...,...,...
45224,2239,E,mp-559415,87.655424
45225,2230,E,mp-1223305,89.773011
45226,1777,E,mp-6036,90.108153
45227,22381,E,mp-1183057,94.462643
